In [ ]:
import os
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import seaborn as sns

In [ ]:
import glob
import os
import pandas as pd

def load_and_combine_csvs(directory, proof_type, device):
    pattern = f"{proof_type.upper()}_multiplication_results_{device.lower()}_*.md"
    files = sorted(glob.glob(os.path.join(directory, pattern)))

    if not files:
        raise FileNotFoundError(f"No files found for pattern: {pattern}")

    all_data = []

    for file in files:
        with open(file, "r") as f:
            lines = f.readlines()

        data_lines = lines[2:]
        rows = []
        for line in data_lines:
            if "|" not in line:
                continue
            parts = [col.strip().strip('`') for col in line.strip().split("|") if col.strip()]
            
            if len(parts) >= 5:
                row = {
                    "Command": parts[0],
                    "Mean [ms]": parts[1],
                    "Min [ms]": parts[2],
                    "Max [ms]": parts[3],
                    "Relative": parts[4],
                    "SourceFile": os.path.basename(file)
                }
                rows.append(row)

        df = pd.DataFrame(rows)
        all_data.append(df)

    combined_df = pd.concat(all_data, ignore_index=True)
    return combined_df


In [ ]:
def extract_input_pairs(df):
    def parse_command(command):
        match = re.search(r"--\s+(\d+)\s+(\d+)$", command)
        return tuple(map(int, match.groups())) if match else (None, None)

    df[['X', 'Y']] = df['Command'].apply(lambda x: pd.Series(parse_command(str(x))))

    # Extract numeric value from "Mean [ms]"
    df['Mean [ms]'] = df['Mean [ms]'].str.extract(r"([\d.]+)").astype(float)

    return df.dropna(subset=['X', 'Y'])



In [ ]:
def create_pivot_table(df):
    return df.pivot_table(index='Y', columns='X', values='Mean [ms]')

In [ ]:
def plot_dual_3d_surfaces(df1, df2, label1="Lambda", label2="Pi", title="Comparison of Mean Execution Time", save_path=None):
    from mpl_toolkits.mplot3d import Axes3D
    import matplotlib.pyplot as plt
    import numpy as np

    pivot1 = create_pivot_table(df1)
    pivot2 = create_pivot_table(df2)

    X1, Y1 = np.meshgrid(pivot1.columns, pivot1.index)
    Z1 = pivot1.values

    X2, Y2 = np.meshgrid(pivot2.columns, pivot2.index)
    Z2 = pivot2.values

    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')

    surf1 = ax.plot_surface(X1, Y1, Z1, alpha=0.7, cmap='viridis', label=label1)
    surf2 = ax.plot_surface(X2, Y2, Z2, alpha=0.7, cmap='plasma', label=label2)

    ax.set_title(title)
    ax.set_xlabel("Input X")
    ax.set_ylabel("Input Y")
    ax.set_zlabel("Mean Time (ms)")

    from matplotlib.lines import Line2D
    custom_lines = [Line2D([0], [0], color='blue', lw=4, label=label1),
                    Line2D([0], [0], color='orange', lw=4, label=label2)]
    # ax.legend(handles=custom_lines)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, format='png')
        print(f"Plot saved to: {save_path}")
    else:
        plt.show()


In [ ]:
df_lambda = extract_input_pairs(load_and_combine_csvs(".", "SNARK", "lambda"))
df_pi = extract_input_pairs(load_and_combine_csvs(".", "SNARK", "pi"))

# Plot comparison
plot_dual_3d_surfaces(
    df_lambda,
    df_pi,
    label1="Lambda",
    label2="Raspberry Pi",
    title="SNARK Performance: Lambda vs Pi",
    save_path="snark_lambda_vs_pi.png"
)